# Deteksi Watermark "sabila" pada 1 Video — Setelah Neural Codec

Notebook demo/sistem deteksi **1 video** (bukan training / evaluasi dataset). Seluruh komponen teknis diambil dari `sabila(1).ipynb` (Neural Encoder–Decoder V2, ECC repetition, Neural Codec, aturan threshold BER).

```
INPUT 1 VIDEO
  → sisipkan watermark "sabila"          (Neural Watermark Encoder, 1x)
  → encode payload dengan ECC            (repetition x3: 48 bit → 144 bit)
  → Neural Codec / kompresi neural       (1x)
  → ekstraksi watermark                  (Neural Watermark Decoder, 1x)
  → ECC decoding / recovery              (majority vote → 48 bit)
  → validasi (BER raw ≤ threshold  AND  ECC valid  AND  payload == target)
  → TERDETEKSI / TIDAK TERDETEKSI
```

**Tidak ada retry:** satu video diproses satu kali — 1× insertion, 1× Neural Codec, 1× ekstraksi. Jika gagal → `TIDAK TERDETEKSI`.

## Payload & ECC (dibedakan secara eksplisit)

| Istilah | Panjang | Isi |
|---|---|---|
| **Payload** | **48 bit** | ASCII `sabila` = `01110011 01100001 01100010 01101001 01101100 01100001` |
| **Raw / ECC bits** | 144 bit | payload diulang 3× (`[payload][payload][payload]`) — inilah yang disisipkan & diekstrak |
| **Recovered payload** | 48 bit | hasil majority-vote 3 salinan dari raw bits hasil ekstraksi |

Teks hasil ekstraksi selalu berasal dari **recovered payload**, bukan dari nilai yang di-hardcode.

## Threshold (dari source)

Source memakai uji binomial: `BER_thr` = ambang dengan *false positive rate* ≤ 10⁻⁶ untuk `n` bit (fungsi `ber_threshold_from_fpr`, disalin apa adanya).
- Run aktual source: `n = WATERMARK_LENGTH = 144` → **k_min = 101, `BER_thr = 0.2986`** (tercetak di output Bagian 10B source).
- Teks markdown & komentar source yang menyebut `0.2031` (n = 64) dan fallback `0.25` sudah usang → **tidak dipakai**. Nilai `0.2986` dipilih karena itulah yang dihitung pipeline deteksi aktual (144 bit).
- CACS (skor kualitas/confidence) di source **tidak menggerbang** status deteksi → tidak dipakai di sini.

**Aturan keputusan:** `detected = (BER_raw ≤ 0.2986) AND (ECC recovery valid) AND (recovered payload == payload target)`.
"ECC recovery valid" = recovered payload terdiri dari 6 karakter ASCII printable (kriteria yang sama dengan `bits_to_text_ecc` di source).

## ⚠️ Peringatan penting: kesehatan bobot model dari source

Log di `sabila(1).ipynb` menunjukkan decoder akhir (fine-tune V2/V3) **collapse**: uji diagnostik source sendiri menghasilkan teks `sabila` untuk video **tanpa watermark**, dan akurasi bit terhadap watermark acak tetap ≈ 0.50 pada seluruh epoch fine-tune V3 yang tercatat di log. Artinya `BER = 0` di 224 video pada source bukan bukti pembacaan watermark.

Notebook ini memuat bobot sesuai instruksi (tidak retrain). **Jalankan Cell 8 (sanity check) sebelum mempercayai status `TERDETEKSI`.** Jika Cell 8 menyatakan FAIL, penyebabnya ada di bobot hasil training, bukan di alur notebook ini; ganti bobot di `WEIGHTS_DIRS` tanpa mengubah kode.

H.264 / H.265 / CACS / evaluasi dataset hanya eksperimen pembanding terpisah di notebook source — **tidak** ada di pipeline ini.

## CELL 1 — Setup / Install Dependencies

Hanya dependency yang dipakai: OpenCV (baca/tulis frame), Gradio (UI), ffmpeg (video lossless FFV1 & preview mp4), Google Drive (lokasi bobot dari source). `torch`, `numpy`, `matplotlib` sudah tersedia di Colab.

In [ ]:
# CELL 1 — Setup / Install Dependencies
import sys, subprocess, shutil

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import cv2  # noqa: F401  (Colab sudah membawa OpenCV)
except ImportError:
    _pip("opencv-python-headless")

_pip("gradio")  # UI (Cell 6)

if shutil.which("ffmpeg") is None:  # dibutuhkan: FFV1 lossless + preview mp4
    subprocess.run(["apt-get", "-y", "install", "ffmpeg"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Google Drive = lokasi bobot yang disimpan notebook source
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Bukan Google Colab -> mount Drive dilewati.")

print("Setup selesai. ffmpeg:", shutil.which("ffmpeg"))

## CELL 2 — Configuration

Semua nilai di bawah berasal dari source (`FRAME_SIZE`, `MAX_FRAMES_PER_VIDEO`, `ECC_REPEAT`, `TARGET_FPR`, arsitektur V2, `quant_levels=16`). Payload diverifikasi dengan `assert` terhadap 48-bit ASCII `sabila`.

**Lokasi bobot** (dicari berurutan di `WEIGHTS_DIRS`):
- `encoder_finetune_v3.pth`, `decoder_finetune_v3.pth` — **wajib ada**, disimpan oleh source (keadaan akhir yang dipakai Part 6–12 source). Ganti `WEIGHTS_VERSION` ke `"finetune_v2"` / `"base"` untuk versi lain.
- `neural_codec.pth` — **tidak pernah disimpan oleh source** (Neural Codec hanya hidup di memori sesi). Notebook ini mencoba tiga jalur berurutan (Cell 3 & 5), tanpa perlu mengubah kode:
  1. **File tersimpan**: jika Anda masih punya sesi lama, `torch.save(neural_codec.state_dict(), "<dir>/neural_codec.pth")` lalu taruh di `WEIGHTS_DIRS`.
  2. **`DATASET_DIR` tersedia**: Cell 3 melatih ulang persis resep source Bagian 4B (≤20 video, 300 frame, 60 epoch), lalu menyimpannya.
  3. **Tidak ada keduanya**: Cell 5 melatih Neural Codec darurat dari **video demo pertama yang Anda upload** (dengan augmentasi flip, 60 epoch), lalu menyimpannya untuk run berikutnya di sesi yang sama. Kompresi hasil jalur ini lebih lemah daripada bootstrap dari 20 video — wajar untuk demo 1 video, dan tetap merupakan Neural Codec asli (bukan rename file/kompresi biasa).

In [ ]:
# CELL 2 — Configuration
import os, time, random, tempfile, html
from math import comb
import numpy as np
import cv2
import torch

# ---------- Target watermark & payload (48 bit) ----------
TARGET_TEXT = "sabila"
TARGET_PAYLOAD_BITS = [int(b) for ch in TARGET_TEXT for b in format(ord(ch), "08b")]
PAYLOAD_BITS = len(TARGET_PAYLOAD_BITS)
assert PAYLOAD_BITS == 48, f"Payload harus 48 bit, dapat {PAYLOAD_BITS}"
_EXPECTED_BINARY = "01110011 01100001 01100010 01101001 01101100 01100001"
assert " ".join("".join(map(str, TARGET_PAYLOAD_BITS[i:i + 8])) for i in range(0, 48, 8)) == _EXPECTED_BINARY

# ---------- ECC: repetition code (source: [FIX #2]) ----------
ECC_REPEAT = 3                                    # [payload][payload][payload]
WATERMARK_LENGTH = PAYLOAD_BITS * ECC_REPEAT      # 144 raw/ECC bits
TARGET_RAW_BITS = TARGET_PAYLOAD_BITS * ECC_REPEAT

# ---------- Threshold (source: Bagian 10B, uji binomial) ----------
TARGET_FPR = 1e-6

def ber_threshold_from_fpr(n_bits, target_fpr=TARGET_FPR):
    """H0: bit hasil ekstraksi ~ Bernoulli(0.5). Pilih k_min terkecil sehingga
    P(K >= k_min | H0) <= target_fpr; BER_thr = (n - k_min) / n. (disalin dari source)"""
    total = 2.0 ** n_bits
    tail, k_min = 0.0, n_bits
    for k in range(n_bits, -1, -1):
        tail += comb(n_bits, k)
        if tail / total <= target_fpr:
            k_min = k
        else:
            break
    return (n_bits - k_min) / n_bits, k_min

BER_THR, K_MIN = ber_threshold_from_fpr(WATERMARK_LENGTH, TARGET_FPR)   # 144 bit -> 0.2986 (k_min=101)

# ---------- Preprocessing & model (source) ----------
FRAME_SIZE = (128, 128)      # (H, W)
MAX_FRAMES = 100             # MAX_FRAMES_PER_VIDEO di source
CHUNK = 16                   # frame per batch inferensi (tidak mengubah hasil: GroupNorm per-sampel)
SEED = 42

# ---------- Paths ----------
SOURCE_OUTPUT_DIR = "/content/drive/MyDrive/Video_data/output"     # OUTPUT_DIR di source
WEIGHTS_DIRS = [SOURCE_OUTPUT_DIR, "/content/weights"]              # dicari berurutan
WEIGHTS_VERSION = "finetune_v3"                                     # "finetune_v3" | "finetune_v2" | "base"
DATASET_DIR = "/content/drive/MyDrive/Video_data/test"              # OPSIONAL: dataset multi-video untuk bootstrap Neural Codec (kualitas terbaik, meniru source)
WORK_DIR = "/content/sabila_demo"
os.makedirs(WORK_DIR, exist_ok=True)

# ---------- Resep bootstrap Neural Codec (source Bagian 4B) — dipakai hanya jika neural_codec.pth belum ada ----------
# Dicoba berurutan (Cell 3): (1) muat neural_codec.pth kalau ada, (2) train dari DATASET_DIR kalau folder itu
# ada, (3) kalau keduanya tidak ada, DITUNDA -> di-train otomatis dari video demo pertama yang diproses
# (Cell 5), memakai augmentasi flip agar tetap ada variasi walau sumbernya cuma 1 video. Opsi (3) ada
# karena sabila(1).ipynb TIDAK PERNAH menyimpan bobot Neural Codec, sehingga tanpa dataset 224 video pun
# notebook ini harus tetap bisa jalan untuk demo 1 video.
CODEC_TRAIN_VIDEOS, CODEC_TRAIN_FRAMES = 20, 300     # dipakai bila DATASET_DIR tersedia
CODEC_BATCH_SIZE, CODEC_EPOCHS, CODEC_LR = 8, 60, 1e-3
CODEC_FALLBACK_EPOCHS = 60                            # dipakai pada bootstrap darurat dari 1 video demo (Cell 5)

# ---------- UI ----------
UI_SHARE = False             # True -> buat public link Gradio

# ---------- Device ----------
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device           : {device}")
print(f"Target text      : '{TARGET_TEXT}'")
print(f"Payload (48 bit) : {' '.join(''.join(map(str, TARGET_PAYLOAD_BITS[i:i+8])) for i in range(0, 48, 8))}")
print(f"ECC              : repetition x{ECC_REPEAT} -> {WATERMARK_LENGTH} raw bits")
print(f"Threshold        : BER_raw <= {BER_THR:.4f}  (k_min={K_MIN}/{WATERMARK_LENGTH}, FPR={TARGET_FPR:g})")
print(f"Bobot            : {WEIGHTS_VERSION} dari {WEIGHTS_DIRS}")

## CELL 3 — Load Existing Model / Existing Weights

Kelas `WatermarkEncoder`, `WatermarkDecoder` (versi V2: `channels=80`, residual `0.18`, blok `conv_mid2`) dan `NeuralCodec` (bottleneck 8, kuantisasi 16 level) **disalin apa adanya** dari source agar `state_dict` kompatibel. Bobot hanya **dimuat** — tidak ada training encoder/decoder.

In [ ]:
# CELL 3 — Load Existing Model / Existing Weights
import torch.nn as nn
import torch.nn.functional as F


# ===== Neural Codec (source Bagian 4B, verbatim) =====
def ste_quantize(z, levels=16):
    z_hard = torch.round(z * levels) / levels
    return z + (z_hard - z).detach()


class NeuralCodec(nn.Module):
    """Compressive autoencoder: bottleneck sempit + kuantisasi (STE)."""
    def __init__(self, bottleneck=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, bottleneck, 3, padding=1),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(bottleneck, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 3, padding=1),
        )

    def forward(self, x, quant_levels=16):
        z = torch.tanh(self.enc(x))
        z = ste_quantize(z, quant_levels)
        return torch.sigmoid(self.dec(z))


# ===== Watermark Encoder / Decoder V2 (source [FIX #3], verbatim) =====
class WatermarkEncoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.wm_length = wm_length
        self.fc_wm = nn.Linear(wm_length, 16 * 8 * 8)
        self.wm_upsample = nn.Sequential(
            nn.ConvTranspose2d(16, 32, 4, stride=2, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.GroupNorm(4, 16), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 8, 4, stride=2, padding=1), nn.GroupNorm(2, 8), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1), nn.GroupNorm(2, 4), nn.ReLU(inplace=True),
        )
        self.conv_in = nn.Sequential(
            nn.Conv2d(3 + 4, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_mid = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_mid2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_out = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, frame, wm_bits):
        B, C, H, W = frame.shape
        wm_feat = self.fc_wm(wm_bits).view(B, 16, 8, 8)
        wm_map = self.wm_upsample(wm_feat)
        x = torch.cat([frame, wm_map], dim=1)
        x = self.conv_in(x)
        x = self.conv_mid(x) + x
        x = self.conv_mid2(x) + x
        residual = torch.tanh(self.conv_out(x)) * 0.18
        watermarked = torch.clamp(frame + residual, 0.0, 1.0)
        return watermarked, residual


class WatermarkDecoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Sequential(
            nn.Linear(channels * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Linear(256, wm_length),
        )

    def forward(self, frame):
        x = self.features(frame)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)


# ===== Lokasi & pemuatan bobot =====
_SUFFIX = {"base": "", "finetune_v2": "_finetune_v2", "finetune_v3": "_finetune_v3"}[WEIGHTS_VERSION]


def _find_weight(fname):
    for d in WEIGHTS_DIRS:
        p = os.path.join(d, fname)
        if os.path.isfile(p):
            return p
    return None


def _weights_save_dir():
    for d in WEIGHTS_DIRS:
        if os.path.isdir(d) and os.access(d, os.W_OK):
            return d
    os.makedirs(WEIGHTS_DIRS[-1], exist_ok=True)
    return WEIGHTS_DIRS[-1]


def _read_frames_simple(path, max_frames):
    cap = cv2.VideoCapture(path)
    out = []
    while len(out) < max_frames:
        ok, f = cap.read()
        if not ok:
            break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        f = cv2.resize(f, (FRAME_SIZE[1], FRAME_SIZE[0]))
        out.append(f.astype(np.float32) / 255.0)
    cap.release()
    return out


def _train_codec_on_frames(codec, frame_pool, epochs, tag):
    """Loop training inti (source Bagian 4B): self-supervised MSE, Adam lr=CODEC_LR, batch CODEC_BATCH_SIZE."""
    data = torch.tensor(np.stack(frame_pool)).permute(0, 3, 1, 2).float().to(device)
    n = data.size(0)
    print(f"[{tag}] {n} frame, {epochs} epoch")
    torch.manual_seed(SEED)
    codec.train()
    opt = torch.optim.Adam(codec.parameters(), lr=CODEC_LR)
    for ep in range(epochs):
        perm = torch.randperm(n)
        ep_loss = 0.0
        for i in range(0, n, CODEC_BATCH_SIZE):
            batch = data[perm[i:i + CODEC_BATCH_SIZE].to(data.device)]
            loss = F.mse_loss(codec(batch), batch)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item() * batch.size(0)
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f"   epoch {ep + 1}/{epochs} recon_loss={ep_loss / n:.5f}")
    codec.eval()
    for p in codec.parameters():
        p.requires_grad_(False)


def _bootstrap_train_neural_codec_from_dataset(codec):
    """Jalur PALING SESUAI SOURCE: 300 frame dari <=20 video di DATASET_DIR, 60 epoch (Bagian 4B).
    Hanya dipanggil kalau DATASET_DIR memang ada."""
    skip = ("_watermarked", "_h264", "_h265", "_neuralcodec")
    vids = sorted(f for f in os.listdir(DATASET_DIR)
                  if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv")) and not any(s in f.lower() for s in skip))
    pool = []
    for v in vids[:CODEC_TRAIN_VIDEOS]:
        pool.extend(_read_frames_simple(os.path.join(DATASET_DIR, v), 100))
        if len(pool) >= CODEC_TRAIN_FRAMES:
            break
    pool = pool[:CODEC_TRAIN_FRAMES]
    if not pool:
        raise RuntimeError(f"Tidak ada frame yang bisa dibaca dari {DATASET_DIR}")
    _train_codec_on_frames(codec, pool, CODEC_EPOCHS, "bootstrap Neural Codec dari DATASET_DIR")
    save_path = os.path.join(_weights_save_dir(), "neural_codec.pth")
    torch.save(codec.state_dict(), save_path)
    print(f"[bootstrap Neural Codec] tersimpan -> {save_path}")
    return save_path


def bootstrap_neural_codec_from_video(codec, video_path):
    """Jalur DARURAT (dipanggil dari Cell 5): dipakai HANYA jika neural_codec.pth tidak ada DAN
    DATASET_DIR juga tidak ada -- sabila(1).ipynb tidak pernah menyimpan bobot Neural Codec, jadi
    tanpa dataset 224 video pun notebook ini harus tetap bisa didemokan dengan 1 video.

    Frame pool diperkaya dengan flip horizontal (augmentasi yang sama dipakai source, fungsi
    augment_batch) supaya tidak melatih pada frame yang identik persis berulang. Kualitas
    kompresi Neural Codec hasil jalur ini TIDAK setara dengan bootstrap dari 20 video/300 frame
    di atas -- ini kompromi yang disengaja agar demo tetap bisa berjalan tanpa dataset."""
    print("[PERINGATAN] neural_codec.pth tidak ada & DATASET_DIR tidak ada -> melatih Neural Codec "
          "darurat dari video demo ini saja (kualitas lebih rendah dari bootstrap 20 video di source).")
    frames, _, _ = extract_frames(video_path, max_frames=MAX_FRAMES)
    if not frames:
        raise ValueError(f"Video demo tidak bisa dibaca untuk bootstrap Neural Codec: {video_path}")
    pool = frames + [f[:, ::-1, :].copy() for f in frames]   # augmentasi flip horizontal (source: augment_batch)
    _train_codec_on_frames(codec, pool, CODEC_FALLBACK_EPOCHS, "bootstrap darurat Neural Codec dari 1 video")
    save_path = os.path.join(_weights_save_dir(), "neural_codec.pth")
    torch.save(codec.state_dict(), save_path)
    print(f"[bootstrap darurat Neural Codec] tersimpan -> {save_path} (dipakai otomatis untuk run berikutnya)")
    return save_path


encoder = WatermarkEncoder().to(device)
decoder = WatermarkDecoder().to(device)
neural_codec = NeuralCodec().to(device)

# --- Encoder & Decoder: WAJIB ada (tidak ada mekanisme bootstrap yang masuk akal untuk ini) ---
_enc_path = _find_weight(f"encoder{_SUFFIX}.pth")
_dec_path = _find_weight(f"decoder{_SUFFIX}.pth")
_missing = [n for n, p in ((f"encoder{_SUFFIX}.pth", _enc_path), (f"decoder{_SUFFIX}.pth", _dec_path)) if p is None]
if _missing:
    raise FileNotFoundError(f"Bobot tidak ditemukan: {_missing}\nDicari di: {WEIGHTS_DIRS}\n"
                            "Pastikan Drive ter-mount (Cell 1) atau upload file .pth ke /content/weights.")

encoder.load_state_dict(torch.load(_enc_path, map_location=device))
decoder.load_state_dict(torch.load(_dec_path, map_location=device))
encoder.eval(); decoder.eval()
for _p in list(encoder.parameters()) + list(decoder.parameters()):
    _p.requires_grad_(False)

# --- Neural Codec: (1) file tersimpan -> (2) DATASET_DIR -> (3) ditunda ke bootstrap darurat di Cell 5 ---
_codec_path = _find_weight("neural_codec.pth")
NEURAL_CODEC_NEEDS_BOOTSTRAP = False
if _codec_path:
    neural_codec.load_state_dict(torch.load(_codec_path, map_location=device))
    neural_codec.eval()
    for _p in neural_codec.parameters():
        _p.requires_grad_(False)
elif os.path.isdir(DATASET_DIR):
    _codec_path = _bootstrap_train_neural_codec_from_dataset(neural_codec)
else:
    NEURAL_CODEC_NEEDS_BOOTSTRAP = True
    print("[INFO] neural_codec.pth tidak ada & DATASET_DIR tidak ada -> Neural Codec akan di-bootstrap "
          "otomatis dari video demo pertama yang diproses (lihat Cell 5).")

print("Encoder      :", _enc_path)
print("Decoder      :", _dec_path)
print("Neural Codec :", _codec_path if _codec_path else "(akan di-bootstrap saat video demo pertama diproses)")
print("\nModel dimuat (encoder/decoder tanpa training). Status TERDETEKSI baru bermakna jika Cell 8 (sanity check) lulus.")

## CELL 4 — Helper Functions

Fungsi reusable: text↔binary, ECC encode/decode, preprocessing frame/video, insertion, Neural Codec, ekstraksi, BER, bit accuracy, validasi.

- `extract_frames`, `frames_to_video_lossless`: dari source Bagian 3. Video ber-watermark & hasil codec disimpan **lossless (FFV1, `.mkv`)** seperti source, agar satu-satunya distorsi terukur berasal dari Neural Codec.
- `ecc_decode` **hanya** memakai bit hasil ekstraksi — payload target tidak dipakai saat decoding (tidak ada kebocoran); target hanya dipakai pada `validate_extraction` untuk perbandingan.
- BER dihitung di dua level: **raw** (144 bit vs 144 bit target; dasar threshold) dan **payload** (48 bit hasil ECC vs 48 bit target).
- Karakter non-printable pada recovered payload ditampilkan `?` (source membuangnya diam-diam sehingga kegagalan tersembunyi).
- `make_preview_mp4` hanya membuat salinan **untuk ditampilkan di browser** (FFV1 tidak bisa diputar). Salinan itu tidak dipakai ekstraksi.

In [ ]:
# CELL 4 — Helper Functions
import subprocess

# ---------------------------------------------------------------- text / binary / ECC
def text_to_bits(text):
    """Teks -> bit ASCII (8 bit / karakter)."""
    return [int(b) for ch in text for b in format(ord(ch), "08b")]


def bits_to_str(bits, group=8):
    s = "".join(str(int(b)) for b in bits)
    return " ".join(s[i:i + group] for i in range(0, len(s), group))


def ecc_encode(payload_bits, repeat=ECC_REPEAT):
    """Repetition code (source [FIX #2]): seluruh payload diulang `repeat` kali secara berurutan."""
    return list(payload_bits) * repeat


def ecc_decode(raw_bits, n_payload=PAYLOAD_BITS, repeat=ECC_REPEAT):
    """Majority-vote antar salinan. Hanya memakai raw_bits hasil ekstraksi.
    Return (payload_bits[np.int], jumlah_posisi_yang_salinannya_tidak_sepakat)."""
    raw = np.asarray(raw_bits).astype(int).flatten()
    if raw.size != n_payload * repeat:
        raise ValueError(f"raw bits harus {n_payload * repeat}, dapat {raw.size}")
    votes = raw.reshape(repeat, n_payload).sum(axis=0)
    payload = (votes * 2 > repeat).astype(int)
    n_disagree = int(np.sum((votes != 0) & (votes != repeat)))
    return payload, n_disagree


def payload_bits_to_text(bits):
    bits = [int(b) for b in bits]
    chars = []
    for i in range(0, len(bits) - len(bits) % 8, 8):
        v = int("".join(map(str, bits[i:i + 8])), 2)
        chars.append(chr(v) if 32 <= v <= 126 else "?")
    return "".join(chars)


def payload_is_valid_text(bits, expected_chars=len(TARGET_TEXT)):
    """ECC recovery dianggap valid bila menghasilkan `expected_chars` karakter ASCII printable (32..126),
    kriteria yang sama dengan bits_to_text_ecc di source."""
    bits = [int(b) for b in bits]
    if len(bits) != expected_chars * 8:
        return False
    return all(32 <= int("".join(map(str, bits[i:i + 8])), 2) <= 126 for i in range(0, len(bits), 8))


def bit_error_rate(bits_true, bits_pred):
    a = np.asarray(bits_true).astype(int).flatten()
    b = np.asarray(bits_pred).astype(int).flatten()
    return float(np.mean(a != b))


def bit_accuracy(bits_true, bits_pred):
    return 1.0 - bit_error_rate(bits_true, bits_pred)


# konsistensi payload/ECC (validasi teknis #1-#2)
assert text_to_bits(TARGET_TEXT) == TARGET_PAYLOAD_BITS
assert ecc_encode(TARGET_PAYLOAD_BITS) == TARGET_RAW_BITS and len(TARGET_RAW_BITS) == WATERMARK_LENGTH
assert list(ecc_decode(TARGET_RAW_BITS)[0]) == TARGET_PAYLOAD_BITS
assert payload_bits_to_text(ecc_decode(TARGET_RAW_BITS)[0]) == TARGET_TEXT


# ---------------------------------------------------------------- validation
def validate_extraction(extracted_raw_bits):
    """Keputusan deteksi dari bit hasil ekstraksi. Tidak ada nilai hardcoded:
    detected = BER_raw<=threshold AND ECC recovery valid AND recovered payload == payload target."""
    raw = np.asarray(extracted_raw_bits).astype(int).flatten()
    payload, n_disagree = ecc_decode(raw)
    ber_raw = bit_error_rate(TARGET_RAW_BITS, raw)
    ber_payload = bit_error_rate(TARGET_PAYLOAD_BITS, payload)
    checks = {
        "ber_raw_le_threshold": bool(ber_raw <= BER_THR),
        "ecc_recovery_valid": bool(payload_is_valid_text(payload)),
        "payload_matches_target": bool(np.array_equal(payload, np.asarray(TARGET_PAYLOAD_BITS))),
    }
    return {
        "detected": bool(all(checks.values())),
        "checks": checks,
        "raw_bits": raw,
        "payload_bits": payload,
        "text": payload_bits_to_text(payload),
        "n_disagree": n_disagree,
        "ber_raw": ber_raw, "bit_accuracy_raw": 1.0 - ber_raw,
        "ber_payload": ber_payload, "bit_accuracy_payload": 1.0 - ber_payload,
    }


# ---------------------------------------------------------------- frame / video preprocessing
def extract_frames(video_path, frame_size=FRAME_SIZE, max_frames=MAX_FRAMES):
    """Baca frame -> RGB, resize (H,W)=frame_size, normalisasi [0,1]. (source Bagian 3)"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps or fps <= 0 or fps > 240:
        fps = 25.0
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = []
    while cap.isOpened() and len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (frame_size[1], frame_size[0]))
        frames.append(frame.astype(np.float32) / 255.0)
    cap.release()
    return frames, fps, (orig_w, orig_h)


def frames_to_video_lossless(frames, out_path, fps, size=None):
    """Simpan frame ([0,1], RGB) sebagai video LOSSLESS (FFV1, .mkv). (source Bagian 3)"""
    h, w = frames[0].shape[:2] if size is None else (size[1], size[0])
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, f in enumerate(frames):
            u8 = np.clip(f * 255.0, 0, 255).astype(np.uint8)
            u8 = cv2.resize(u8, (w, h))
            cv2.imwrite(os.path.join(tmpdir, f"f_{i:05d}.png"), cv2.cvtColor(u8, cv2.COLOR_RGB2BGR))
        cmd = ["ffmpeg", "-y", "-framerate", str(fps), "-i", os.path.join(tmpdir, "f_%05d.png"),
               "-c:v", "ffv1", out_path]
        proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0 or not os.path.isfile(out_path):
        raise RuntimeError("ffmpeg gagal menulis video lossless:\n" + proc.stderr[-500:])
    return out_path


def make_preview_mp4(src_path, dst_path):
    """HANYA untuk tampilan browser (FFV1/.mkv tidak bisa diputar). Tidak dipakai ekstraksi."""
    cmd = ["ffmpeg", "-y", "-i", src_path, "-c:v", "libx264", "-crf", "18",
           "-pix_fmt", "yuv420p", "-movflags", "+faststart", dst_path]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0 or not os.path.isfile(dst_path):
        raise RuntimeError("ffmpeg gagal membuat preview mp4:\n" + proc.stderr[-500:])
    return dst_path


def psnr_db(a, b):
    mse = float(np.mean((np.asarray(a, np.float64) - np.asarray(b, np.float64)) ** 2))
    return float("inf") if mse < 1e-12 else 10.0 * float(np.log10(1.0 / mse))


# ---------------------------------------------------------------- model wrappers (encoder / Neural Codec / decoder)
def _chunks(n, size=CHUNK):
    for i in range(0, n, size):
        yield slice(i, min(i + size, n))


def _to_tensor(x):
    return torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).permute(0, 3, 1, 2).contiguous().to(device)


def _to_numpy(t):
    return t.permute(0, 2, 3, 1).detach().cpu().numpy()


def insert_watermark(frames, raw_bits):
    """Neural Watermark Encoder: watermark (raw/ECC bits) yang SAMA disisipkan ke semua frame.
    Return (watermarked (N,H,W,3), residual (N,H,W,3))."""
    bits = torch.tensor(np.asarray(raw_bits, dtype=np.float32), device=device).unsqueeze(0)
    wm_out, res_out = [], []
    with torch.no_grad():
        for s in _chunks(len(frames)):
            x = _to_tensor(frames[s])
            w, r = encoder(x, bits.repeat(x.size(0), 1))
            wm_out.append(_to_numpy(w)); res_out.append(_to_numpy(r))
    return np.concatenate(wm_out), np.concatenate(res_out)


def neural_codec_compress(frames):
    """Neural Codec (compressive autoencoder + kuantisasi 16 level) -> frame rekonstruksi."""
    out = []
    with torch.no_grad():
        for s in _chunks(len(frames)):
            out.append(_to_numpy(neural_codec(_to_tensor(frames[s]))))
    return np.concatenate(out)


def extract_probs(frames):
    """Neural Watermark Decoder: probabilitas bit per frame (N, 144)."""
    out = []
    with torch.no_grad():
        for s in _chunks(len(frames)):
            out.append(torch.sigmoid(decoder(_to_tensor(frames[s]))).cpu().numpy())
    return np.concatenate(out)


print("Helper functions siap. Konsistensi payload/ECC lolos assert:",
      f"payload={len(TARGET_PAYLOAD_BITS)} bit, raw={len(TARGET_RAW_BITS)} bit.")

## CELL 5 — Single Video Pipeline

`process_video(video_path) -> result` menjalankan alur **satu kali**, tanpa loop/retry:

`video → watermark insertion (1x) → Neural Codec (1x) → extraction (1x) → ECC decoding → validation → result`

- Ekstraksi membaca **file hasil Neural Codec** (bukan frame di memori sebelum kompresi). Neural Codec dipastikan benar-benar mengubah frame (`codec_mean_abs_change > 0`, jika tidak → error).
- Video ber-watermark → dibaca kembali dari file lossless → Neural Codec → ditulis lossless → dibaca kembali oleh decoder (mengikuti source Part 6–8).
- Seperti source, video diproses pada `128×128`, maksimal `100` frame.
- `_run_pipeline(..., embed_text=None)` (tanpa insertion) dan `embed_text="…"` (payload lain) **hanya** dipakai Cell 8 untuk sanity check; alur utama selalu `embed_text="sabila"`.
- Hasil terdiri dari `ber_payload/bit_accuracy_payload` (48 bit) dan `ber_raw/bit_accuracy_raw` (144 bit, dasar threshold), plus `checks` per kondisi keputusan.

In [ ]:
# CELL 5 — Single Video Pipeline
LAST_RESULT = None
LAST_VIDEO_PATH = None


def _run_pipeline(video_path, embed_text=TARGET_TEXT):
    """1x insertion -> 1x Neural Codec -> 1x extraction -> ECC -> validasi. Tanpa retry.
    embed_text=None  : tanpa insertion (kontrol untuk Cell 8).
    embed_text=str   : payload 6 karakter (alur utama = TARGET_TEXT)."""
    t0 = time.time()
    run_dir = tempfile.mkdtemp(prefix="run_", dir=WORK_DIR)

    # --- preprocessing ---
    frames, fps, orig_size = extract_frames(video_path)
    if not frames:
        raise ValueError("Video tidak bisa dibaca / tidak memiliki frame.")
    frames_np = np.stack(frames).astype(np.float32)
    n = len(frames_np)

    # --- (1) watermark insertion: payload 48 bit -> ECC 144 bit -> encoder ---
    if embed_text is None:
        wm_np, residual, embedded_raw = frames_np, None, None
    else:
        payload = text_to_bits(embed_text)
        if len(payload) != PAYLOAD_BITS:
            raise ValueError(f"embed_text harus {PAYLOAD_BITS // 8} karakter ASCII")
        embedded_raw = ecc_encode(payload)
        wm_np, residual = insert_watermark(frames_np, embedded_raw)
    wm_path = os.path.join(run_dir, "watermarked.mkv")
    frames_to_video_lossless(list(wm_np), wm_path, fps)

    # --- (2) Neural Codec: baca video ber-watermark dari file -> kompresi neural -> tulis file ---
    global NEURAL_CODEC_NEEDS_BOOTSTRAP
    if NEURAL_CODEC_NEEDS_BOOTSTRAP:
        # Jalur darurat: sabila(1).ipynb tidak pernah menyimpan bobot Neural Codec, dan DATASET_DIR
        # tidak tersedia di sesi ini -> latih sekali dari video demo pertama (lihat Cell 3), lalu
        # tandai selesai supaya run berikutnya di sesi yang sama langsung memakai bobot ini.
        bootstrap_neural_codec_from_video(neural_codec, video_path)
        NEURAL_CODEC_NEEDS_BOOTSTRAP = False

    wm_disk, fps_wm, _ = extract_frames(wm_path, max_frames=n)
    wm_disk = np.stack(wm_disk).astype(np.float32)
    comp_np = neural_codec_compress(wm_disk)
    codec_change = float(np.abs(comp_np - wm_disk).mean())
    if codec_change <= 0.0:
        raise RuntimeError("Neural Codec tidak mengubah frame sama sekali -> kompresi tidak berjalan.")
    comp_path = os.path.join(run_dir, "neural_codec.mkv")
    frames_to_video_lossless(list(comp_np), comp_path, fps_wm)

    # --- (3) extraction: dari FILE hasil Neural Codec ---
    comp_disk, _, _ = extract_frames(comp_path, max_frames=n)
    probs = extract_probs(np.stack(comp_disk).astype(np.float32))       # (N, 144)
    avg_probs = probs.mean(axis=0)                                        # agregasi antar-frame (source Bagian 8)
    raw_bits = (avg_probs > 0.5).astype(int)                              # 144 raw/ECC bits

    # --- (4) ECC decoding + (5) validasi ---
    val = validate_extraction(raw_bits)
    per_frame_ber = ((probs > 0.5).astype(int) != np.asarray(TARGET_RAW_BITS)).mean(axis=1)

    debug = None
    if residual is not None:
        idx = sorted(set(np.linspace(0, n - 1, num=min(4, n)).astype(int).tolist()))
        debug = {"idx": idx, "original": frames_np[idx], "watermarked": wm_np[idx], "residual": residual[idx]}

    return {
        "detected": val["detected"],
        "status": "TERDETEKSI" if val["detected"] else "TIDAK TERDETEKSI",
        "target_text": TARGET_TEXT,
        "extracted_text": val["text"],
        "target_bits": bits_to_str(TARGET_PAYLOAD_BITS),
        "extracted_bits": bits_to_str(val["payload_bits"]),
        "target_raw_bits": bits_to_str(TARGET_RAW_BITS),
        "extracted_raw_bits": bits_to_str(val["raw_bits"]),
        "ber_payload": val["ber_payload"], "bit_accuracy_payload": val["bit_accuracy_payload"],
        "ber_raw": val["ber_raw"], "bit_accuracy_raw": val["bit_accuracy_raw"],
        "threshold": BER_THR,
        "threshold_info": f"BER raw (144 bit) <= {BER_THR:.4f}  [uji binomial, k_min={K_MIN}/{WATERMARK_LENGTH}, FPR={TARGET_FPR:g}]",
        "checks": val["checks"],
        "ecc_positions_disagree": val["n_disagree"],
        "input_video": video_path,
        "watermarked_video": wm_path,
        "compressed_video": comp_path,
        "embedded_text": embed_text,
        "n_frames": n, "fps": float(fps), "original_size": orig_size,
        "per_frame_ber_mean": float(per_frame_ber.mean()),
        "per_frame_ber_min": float(per_frame_ber.min()),
        "per_frame_ber_max": float(per_frame_ber.max()),
        "mean_bit_confidence": float(np.mean(np.abs(avg_probs - 0.5) * 2)),
        "codec_mean_abs_change": codec_change,
        "codec_psnr_db": psnr_db(wm_disk, comp_np),
        "elapsed_s": time.time() - t0,
        "_raw_bits_array": val["raw_bits"],
        "_debug": debug,
    }


def process_video(video_path):
    """Pipeline utama untuk 1 video. Satu kali proses, tidak ada retry."""
    global LAST_RESULT, LAST_VIDEO_PATH
    LAST_VIDEO_PATH = video_path
    LAST_RESULT = _run_pipeline(video_path, embed_text=TARGET_TEXT)
    return LAST_RESULT


print("process_video(video_path) siap. Contoh:  r = process_video('/content/video.mp4'); print(r['status'])")

## CELL 6 — Simple UI (Gradio)

Upload 1 video → **Proses** → tampil: video input, video ber-watermark, video hasil Neural Codec, status deteksi, teks target/hasil, binary 48 bit target/recovery, BER, bit accuracy, threshold, dan info debug ekstraksi (termasuk saat gagal).

Video pada UI adalah salinan preview H.264 (hanya agar bisa diputar di browser); ekstraksi memakai file lossless. Hasil pada resolusi kerja `128×128` seperti source.

In [ ]:
# CELL 6 — Simple UI
import gradio as gr

_PLACEHOLDER = "<div style='padding:14px;border-radius:8px;background:#eee;color:#555;font-size:18px;text-align:center'>Belum ada hasil</div>"


def _status_html(text, ok):
    color = "#1b8a3a" if ok else "#c62828"
    return (f"<div style='padding:14px;border-radius:8px;background:{color};color:white;"
            f"font-size:26px;font-weight:bold;text-align:center'>{html.escape(text)}</div>")


def _debug_text(r):
    c = r["checks"]
    return "\n".join([
        f"Kondisi keputusan  : BER_raw<=threshold={c['ber_raw_le_threshold']} | ECC valid={c['ecc_recovery_valid']} | payload==target={c['payload_matches_target']}",
        f"Raw bits target (144)    : {r['target_raw_bits']}",
        f"Raw bits terekstrak (144): {r['extracted_raw_bits']}",
        f"Posisi payload yang salinan ECC-nya tidak sepakat: {r['ecc_positions_disagree']}/{PAYLOAD_BITS}",
        f"BER per frame (vs raw target): mean={r['per_frame_ber_mean']:.4f} min={r['per_frame_ber_min']:.4f} max={r['per_frame_ber_max']:.4f}",
        f"Rata-rata confidence bit (|p-0.5|*2): {r['mean_bit_confidence']:.3f}",
        f"Neural Codec: PSNR watermarked->compressed = {r['codec_psnr_db']:.2f} dB, mean |perubahan| = {r['codec_mean_abs_change']:.5f}",
        f"Frame diproses: {r['n_frames']} @ {r['fps']:.2f} fps | ukuran asli {r['original_size']} -> {FRAME_SIZE[1]}x{FRAME_SIZE[0]} | waktu {r['elapsed_s']:.1f}s",
        "Catatan: TERDETEKSI hanya valid jika Cell 8 (sanity check) lulus untuk bobot yang dipakai.",
    ])


def ui_process(video_path):
    if not video_path:
        return (None, None, _status_html("Upload 1 video dulu", False),
                TARGET_TEXT, "", bits_to_str(TARGET_PAYLOAD_BITS), "", "", "", "", "")
    try:
        r = process_video(video_path)          # 1 video, 1x proses, tanpa retry
        wm_prev = make_preview_mp4(r["watermarked_video"], os.path.join(os.path.dirname(r["watermarked_video"]), "watermarked_preview.mp4"))
        cp_prev = make_preview_mp4(r["compressed_video"], os.path.join(os.path.dirname(r["compressed_video"]), "neural_codec_preview.mp4"))
    except Exception as e:  # tampilkan error apa adanya
        return (None, None, _status_html("ERROR", False), TARGET_TEXT, "", bits_to_str(TARGET_PAYLOAD_BITS), "", "", "", "", f"{type(e).__name__}: {e}")
    return (
        wm_prev, cp_prev,
        _status_html(r["status"], r["detected"]),
        r["target_text"], r["extracted_text"],
        r["target_bits"], r["extracted_bits"],
        f"Payload (48 bit, setelah ECC): {r['ber_payload']:.4f}\nRaw ECC (144 bit, dasar threshold): {r['ber_raw']:.4f}",
        f"Payload (48 bit): {r['bit_accuracy_payload']:.2%}\nRaw ECC (144 bit): {r['bit_accuracy_raw']:.2%}",
        r["threshold_info"],
        _debug_text(r),
    )


try:
    demo.close()
except Exception:
    pass

with gr.Blocks(title="Deteksi Watermark 'sabila'") as demo:
    gr.Markdown("# Deteksi Watermark `sabila` setelah Neural Codec\nUpload 1 video → **Proses**. Satu video diproses satu kali (tanpa retry).")
    with gr.Row():
        in_video = gr.Video(label="1. Video input / preview")
        with gr.Column():
            btn = gr.Button("Proses", variant="primary")
            gr.Markdown("**4. Status deteksi**")
            status = gr.HTML(_PLACEHOLDER)
    with gr.Row():
        out_wm = gr.Video(label="2. Video setelah watermark disisipkan", interactive=False)
        out_cp = gr.Video(label="3. Video hasil Neural Codec", interactive=False)
    with gr.Row():
        t_target = gr.Textbox(label="5. Teks target", value=TARGET_TEXT, interactive=False)
        t_extracted = gr.Textbox(label="6. Teks hasil ekstraksi (recovered payload)", interactive=False)
    with gr.Row():
        b_target = gr.Textbox(label="7. Binary target (48 bit)", value=bits_to_str(TARGET_PAYLOAD_BITS), interactive=False)
        b_extracted = gr.Textbox(label="8. Binary hasil recovery (48 bit)", interactive=False)
    with gr.Row():
        m_ber = gr.Textbox(label="9. BER", lines=2, interactive=False)
        m_acc = gr.Textbox(label="10. Bit accuracy", lines=2, interactive=False)
        m_thr = gr.Textbox(label="11. Threshold", lines=2, interactive=False)
    with gr.Accordion("12. Info ekstraksi / debug", open=True):
        dbg = gr.Textbox(label="Debug", lines=10, interactive=False)
    btn.click(ui_process, inputs=in_video,
              outputs=[out_wm, out_cp, status, t_target, t_extracted, b_target, b_extracted, m_ber, m_acc, m_thr, dbg])

demo.launch(share=UI_SHARE, debug=False, prevent_thread_lock=True, allowed_paths=[WORK_DIR])   # non-blocking agar Cell 7 & 8 tetap bisa dijalankan

## CELL 7 — Optional Debug Visualization

**Debug saja — bukan bagian keputusan deteksi dan bukan bukti detection.** Menampilkan frame asli, frame ber-watermark, peta intensitas residual, dan zona konsentrasi terkuat (top 15% intensitas) dari hasil `process_video` terakhir (via UI atau pemanggilan langsung). Watermark pada encoder ini tersebar di seluruh frame; kotak hijau hanyalah wilayah residual terbesar.

In [ ]:
# CELL 7 — Optional Debug Visualization (bukan bagian keputusan deteksi)
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as patches


def residual_to_heatmap(residual):
    mag = np.abs(residual).sum(axis=-1)
    mag_norm = (mag - mag.min()) / (mag.max() - mag.min() + 1e-8)
    return cm.inferno(mag_norm)[..., :3], mag_norm


def strongest_zone_bbox(mag_norm, top_percent=15, min_size=12):
    mask = mag_norm >= np.percentile(mag_norm, 100 - top_percent)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x0, x1, y0, y1 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())
    if x1 - x0 < min_size:
        cx = (x0 + x1) // 2
        x0, x1 = max(0, cx - min_size // 2), min(mag_norm.shape[1] - 1, cx + min_size // 2)
    if y1 - y0 < min_size:
        cy = (y0 + y1) // 2
        y0, y1 = max(0, cy - min_size // 2), min(mag_norm.shape[0] - 1, cy + min_size // 2)
    return x0, y0, x1, y1


def show_debug_visualization(result=None, top_percent=15):
    result = result or LAST_RESULT
    if not result or not result.get("_debug"):
        print("Belum ada hasil dengan watermark. Jalankan UI (Cell 6) atau process_video(...) dulu.")
        return
    d = result["_debug"]
    fig, axes = plt.subplots(len(d["idx"]), 3, figsize=(10, 3.3 * len(d["idx"])), squeeze=False)
    for r, i in enumerate(d["idx"]):
        heat, mag = residual_to_heatmap(d["residual"][r])
        box = strongest_zone_bbox(mag, top_percent)
        axes[r, 0].imshow(d["original"][r])
        axes[r, 1].imshow(np.clip(d["watermarked"][r], 0, 1))
        axes[r, 2].imshow(heat)
        if box:
            x0, y0, x1, y1 = box
            for c in (1, 2):
                axes[r, c].add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="lime", facecolor="none"))
        for c in range(3):
            axes[r, c].axis("off")
        axes[r, 0].set_title(f"Asli (frame {i + 1})", fontsize=9)
        axes[r, 1].set_title("Ber-watermark + zona terkuat", fontsize=9)
        axes[r, 2].set_title("Peta intensitas residual", fontsize=9)
    plt.suptitle("DEBUG ONLY — bukan dasar keputusan deteksi", fontsize=11)
    plt.tight_layout(); plt.show()


show_debug_visualization()

## CELL 8 — Optional Diagnostic: Sanity Check False Positive

Bukan bagian alur pengguna. Memastikan status `TERDETEKSI` benar-benar berasal dari watermark yang terbaca, bukan dari decoder yang selalu mengeluarkan `sabila`. Tiga kasus, masing-masing melewati pipeline yang sama (1× insertion bila ada, 1× Neural Codec, 1× ekstraksi):

| Kasus | Input ke pipeline | Hasil yang diharapkan |
|---|---|---|
| A | video **tanpa watermark** | `TIDAK TERDETEKSI` |
| B | watermark **payload lain** (`qwerty`) | `TIDAK TERDETEKSI` terhadap `sabila`; teks terekstrak ≈ `qwerty` |
| C | watermark `sabila` (alur utama) | `TERDETEKSI` |

Kesimpulan: **FAIL** bila A atau B terdeteksi sebagai `sabila`, atau bila bit terekstrak identik di ketiga kasus (decoder tidak bergantung pada input). Sumber video: `SANITY_VIDEO` → video terakhir dari UI → klip sintetis otomatis.

In [ ]:
# CELL 8 — Optional Diagnostic: sanity check false positive
SANITY_VIDEO = ""            # isi path video, atau kosongkan (pakai video terakhir dari UI / klip sintetis)
SANITY_OTHER_TEXT = "qwerty"  # payload lain, harus 6 karakter ASCII


def make_synthetic_video(path, n=30, fps=25, size=(160, 120)):
    w, h = size
    wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    xx, yy = np.meshgrid(np.linspace(0, 1, w), np.linspace(0, 1, h))
    rng = np.random.RandomState(0)
    for t in range(n):
        img = np.stack([xx, yy, (xx + yy + t / n) % 1.0], axis=-1)
        img = (img * 200 + rng.rand(h, w, 3) * 20).astype(np.uint8)
        cv2.circle(img, (int(w * (0.2 + 0.6 * t / n)), h // 2), 15, (255, 255, 255), -1)
        wr.write(img)
    wr.release()
    return path


def run_sanity_check(video_path=None, other_text=SANITY_OTHER_TEXT):
    video_path = video_path or SANITY_VIDEO or LAST_VIDEO_PATH
    if not video_path:
        video_path = make_synthetic_video(os.path.join(WORK_DIR, "synthetic_sanity.mp4"))
        print("Memakai klip sintetis:", video_path)
    else:
        print("Memakai video:", video_path)

    cases = [("A. tanpa watermark", None),
             (f"B. payload lain '{other_text}'", other_text),
             (f"C. watermark target '{TARGET_TEXT}'", TARGET_TEXT)]
    res = {}
    for name, embed in cases:
        r = _run_pipeline(video_path, embed_text=embed)      # masing-masing 1x, tanpa retry
        res[name[0]] = r
        line = (f"{name:<32} -> {r['status']:<17} teks='{r['extracted_text']}'  "
                f"BER_raw(vs sabila)={r['ber_raw']:.4f}  BER_payload={r['ber_payload']:.4f}")
        if embed and embed != TARGET_TEXT:
            own = bit_accuracy(ecc_encode(text_to_bits(embed)), r["_raw_bits_array"])
            line += f"  | akurasi thd payload yg disisipkan sendiri: {own:.2%}"
        print(line)

    A, B, C = res["A"], res["B"], res["C"]
    fp = A["detected"] or B["detected"]
    identical = np.array_equal(A["_raw_bits_array"], C["_raw_bits_array"]) and np.array_equal(B["_raw_bits_array"], C["_raw_bits_array"])
    print(f"\nBit terekstrak identik di A, B, dan C? {identical}")
    print(f"Threshold BER_raw = {BER_THR:.4f}; BER_raw acak diharapkan ~0.5")
    if fp or identical:
        print("\n>>> VERDICT: FAIL — detector menghasilkan 'sabila' / TERDETEKSI tanpa bukti watermark "
              "(decoder tidak bergantung pada isi video).\n    Status TERDETEKSI dari model ini TIDAK bisa dipercaya. "
              "Penyebab ada pada bobot hasil training, bukan pada alur notebook ini.")
    elif C["detected"]:
        print("\n>>> VERDICT: PASS — kontrol tidak terdeteksi, watermark target terdeteksi.")
    else:
        print("\n>>> VERDICT: tidak false-positive, tetapi watermark target juga TIDAK terbaca pada video ini (recall rendah).")
    return res


sanity_results = run_sanity_check()

## Checklist validasi teknis (lokasi di kode)

| # | Validasi | Dijamin oleh |
|---|---|---|
| 1 | Payload target 48 bit | `assert` di Cell 2 (`PAYLOAD_BITS == 48` + string biner) |
| 2 | ECC tidak mengubah definisi payload | Cell 4: `ecc_encode/ecc_decode` round-trip `assert`; raw 144 ≠ payload 48 |
| 3 | Raw bits dibedakan dari recovered payload | `extracted_raw_bits` vs `extracted_bits`; BER/akurasi terpisah (`*_raw`, `*_payload`) |
| 4 | BER terhadap bit yang benar | raw vs `TARGET_RAW_BITS`, payload vs `TARGET_PAYLOAD_BITS` |
| 5 | Teks dari bit hasil ekstraksi/ECC | `payload_bits_to_text(ecc_decode(bit terekstrak))` |
| 6 | TRUE/FALSE tidak hardcoded | `validate_extraction`: `all(checks)` dari data |
| 7 | Tidak ada retry | `_run_pipeline` linear, tanpa loop ulang |
| 8 | Hanya 1 video | `process_video(video_path)` |
| 9 | Neural Codec benar-benar jalan | `neural_codec_compress` + error jika `codec_mean_abs_change == 0`; ekstraksi membaca file hasil codec |
| 10 | Bobot sama dengan source | Cell 3: kelas verbatim + `load_state_dict` dari file `.pth` source (Neural Codec: lihat catatan Cell 2) |
| 11 | Gagal → FALSE | tiap kondisi keputusan independen; Cell 8 kasus A/B |
| 12 | Tanpa kebocoran target | `ecc_decode` tidak menerima target; target hanya pembanding di `validate_extraction` |

Hasil aktual bergantung pada bobot yang dimuat — lihat peringatan di awal notebook dan hasil Cell 8.